# Lecture 6 Demo: Parameterized Quantum Circuits and Variational Objectives

Computational companion notebook for Lecture 6.

Convention reminder: each qubit starts in $\lvert 0\rangle$ by default. To prepare a different basis state, apply gates such as $X$ before the main circuit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import minimize

from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp, Statevector

import pennylane as qml

## Demo 1. Encoding block versus ansatz block

We separate data encoding and trainable layers:
$$
\lvert\psi(x,\theta)\rangle = U_{\mathrm{ansatz}}(\theta)U_{\mathrm{enc}}(x)\lvert 0\rangle.
$$

In [ ]:
def one_qubit_pqc_state(x: float, theta: float) -> np.ndarray:
    qc = QuantumCircuit(1)
    qc.ry(x, 0)         # encoding
    qc.rz(theta, 0)     # ansatz
    return Statevector.from_instruction(qc).data

print(np.round(one_qubit_pqc_state(0.7, 0.2), 6))

## Demo 2. Cost as expectation value

For $\lvert\psi(\theta)\rangle = R_y(\theta)\lvert 0\rangle$ and $O=Z$, the objective is
$$
C(\theta) = \langle Z \rangle = \cos(\theta).
$$

In [ ]:
grid = np.linspace(0.0, 2 * np.pi, 200)
cost = np.cos(grid)

plt.figure(figsize=(6, 3))
plt.plot(grid, cost, label=r"$\langle Z \rangle = \cos(\theta)$")
plt.xlabel(r"$\theta$")
plt.ylabel("cost")
plt.legend()
plt.tight_layout()
plt.show()

## Demo 3. Pauli-sum objective decomposition in Qiskit

We evaluate a two-qubit objective
$$
H = Z\otimes Z + 0.5(X\otimes I + I\otimes X)
$$
as one observable.

In [ ]:
estimator = StatevectorEstimator()
hamiltonian = SparsePauliOp.from_list([
    ("ZZ", 1.0),
    ("XI", 0.5),
    ("IX", 0.5),
])

def two_qubit_ansatz(params: np.ndarray) -> QuantumCircuit:
    qc = QuantumCircuit(2)
    qc.ry(float(params[0]), 0)
    qc.ry(float(params[1]), 1)
    qc.cx(0, 1)
    qc.ry(float(params[2]), 0)
    qc.ry(float(params[3]), 1)
    return qc

val = estimator.run([(two_qubit_ansatz(np.array([0.2, 0.4, -0.1, 0.3])), hamiltonian)]).result()[0].data.evs
print(f"Objective value: {float(val):.6f}")

## Demo 4. One-qubit toy VQE with analytic reference

For $H=Z$ and ansatz $R_y(\theta)\lvert 0\rangle$, the exact minimum is $-1$ at $\theta=\pi$.

In [ ]:
def energy_z(theta: float) -> float:
    return float(np.cos(theta))

result = minimize(lambda t: energy_z(float(t[0])), x0=np.array([0.1]), method="COBYLA")
print("theta*:", float(result.x[0]))
print("E(theta*):", float(result.fun))
print("Reference optimum: theta=pi, E=-1")

## Demo 5. Optimizer comparison on the same objective

We compare a few optimizers on the same two-qubit Hamiltonian objective.

In [ ]:
def objective(params: np.ndarray) -> float:
    qc = two_qubit_ansatz(params)
    return float(estimator.run([(qc, hamiltonian)]).result()[0].data.evs)

x0 = np.array([0.2, -0.3, 0.4, -0.1], dtype=float)
methods = ["COBYLA", "Nelder-Mead", "Powell"]
results = {}

for method in methods:
    hist = []

    def cb(xk):
        hist.append(objective(np.array(xk, dtype=float)))

    res = minimize(objective, x0=x0, method=method, callback=cb, options={"maxiter": 60})
    if not hist:
        hist.append(float(res.fun))
    results[method] = hist

plt.figure(figsize=(6, 3))
for method, hist in results.items():
    plt.plot(hist, label=method)
plt.xlabel("iteration")
plt.ylabel("energy estimate")
plt.legend()
plt.tight_layout()
plt.show()

## Demo 6. Short PennyLane cross-check

The same one-qubit $\langle Z\rangle$ objective is computed with PennyLane.

In [ ]:
dev = qml.device("default.qubit", wires=1)

@qml.qnode(dev)
def pl_cost(theta):
    qml.RY(theta, wires=0)
    return qml.expval(qml.PauliZ(0))

for theta in [0.0, np.pi / 3, np.pi]:
    print(f"theta={theta:.3f} | PennyLane={pl_cost(theta):.6f} | exact={np.cos(theta):.6f}")

## Summary

Variational quantum algorithms train parameters of a circuit by minimizing expectation-value objectives. The architecture controls what can be represented, while the optimizer and shot budget control what can be found in practice.